In [2]:
import pandas as pd
from sklearn.linear_model import LinearRegression

laps = pd.read_csv("../data/processed/laps_2026.csv")
races = laps[laps["session_name"] == "Race"]
g = races[races["location"] == "Melbourne"]

In [3]:
X = pd.get_dummies(g["driver_number"], prefix = "driver", dtype = float)
X["lap_number"] = g["lap_number"]
X["tyre_age"] = g["tyre_age"]
y = g["lap_duration"]

X.head()

,driver_1,driver_3,driver_5,driver_6,driver_10,driver_11,driver_12,driver_14,driver_16,driver_18,...,driver_31,driver_41,driver_43,driver_44,driver_55,driver_63,driver_77,driver_87,lap_number,tyre_age
3515,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,1
3516,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2,1
3517,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2,1
3518,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,1
3519,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2,1


In [4]:
model = LinearRegression().fit(X,y)
coefs = dict(zip(X.columns, model.coef_))
print(coefs["lap_number"], coefs["tyre_age"], model.score(X, y))

-0.05067822179403747 0.037916742522311364 0.7673511613103566


In [5]:
def fit_race_correction(g):

    X = pd.get_dummies(g["driver_number"], prefix = "driver", dtype = float)
    X["lap_number"] = g["lap_number"]
    X["tyre_age"] = g["tyre_age"]
    y = g["lap_duration"]

    model = LinearRegression().fit(X,y)
    coefs = dict(zip(X.columns, model.coef_))

    return {
        "location": g["location"].iloc[0],
        "fuel": coefs["lap_number"],
        "deg": coefs["tyre_age"],
        "r2": model.score(X,y),
    }

In [6]:
g = races[races["location"] == "Melbourne"]
fit_race_correction(g)

{'location': 'Melbourne',
 'fuel': np.float64(-0.05067822179403747),
 'deg': np.float64(0.037916742522311364),
 'r2': 0.7673511613103566}

In [7]:
results = []

for _,g in races.groupby("session_key"):
    results.append(fit_race_correction(g))

results = pd.DataFrame(results)
print(results)


             location      fuel       deg        r2
0           Melbourne -0.050678  0.037917  0.767351
1            Shanghai -0.059084  0.020191  0.827675
2              Suzuka -0.071541  0.035022  0.836185
3       Miami Gardens -0.029846  0.035105  0.784132
4            Montréal -0.033923 -0.022195  0.731443
5         Monte Carlo -0.012454  0.025703  0.137222
6           Barcelona -0.059258  0.129425  0.820028
7           Spielberg -0.037714  0.076543  0.833093
8         Silverstone -0.060449  0.056132  0.896356
9   Spa-Francorchamps -0.061075  0.050215  0.862867
10           Budapest -0.049709  0.065859  0.802476
11          Zandvoort -0.044764  0.042657  0.728800
12              Monza -0.056201  0.013619  0.511651
13             Madrid -0.078584  0.004827  0.788295
14               Baku -0.067945 -0.005665  0.821312
15         Marina Bay -0.043164  0.023615  0.720159
16             Austin -0.063526  0.060091  0.850446
17        Mexico City -0.028146  0.029505  0.786282
18          

In [8]:
results[["fuel", "deg"]].median()

fuel   -0.056201
deg     0.029505
dtype: float64

In [9]:
FUEL = -0.056 
DEG = 0.030

lr = pd.read_csv("../data/processed/long_runs_2026.csv")
display(lr)

,meeting_key,session_key,driver_number,lap_number,date_start,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,...,compound,tyre_age_at_start,tyre_age,is_pit_in_lap,is_neutralised,is_deleted,stint_best,session_name,location,laps_in_stint
0,1279,11228,44,2,2026-03-06T05:01:54.720000+00:00,28.540,17.674,35.292,264.0,291.0,...,HARD,0,1,False,False,False,80.637,Practice 2,Melbourne,5
1,1279,11228,44,5,2026-03-06T05:07:01.178000+00:00,28.428,17.657,34.818,241.0,302.0,...,HARD,0,4,False,False,False,80.637,Practice 2,Melbourne,5
2,1279,11228,44,7,2026-03-06T05:10:23.405000+00:00,28.415,17.981,34.959,240.0,284.0,...,HARD,0,6,False,False,False,80.637,Practice 2,Melbourne,5
3,1279,11228,44,10,2026-03-06T05:15:26.902000+00:00,28.421,17.786,34.694,242.0,288.0,...,HARD,0,9,False,False,False,80.637,Practice 2,Melbourne,5
4,1279,11228,44,12,2026-03-06T05:18:56.125000+00:00,28.241,17.606,34.790,259.0,291.0,...,HARD,0,11,False,False,False,80.637,Practice 2,Melbourne,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1882,1299,11406,77,30,2026-11-06T19:59:27.434000+00:00,19.464,40.269,16.901,318.0,227.0,...,SOFT,6,11,False,False,False,76.244,Practice 2,São Paulo,5
1883,1299,11406,23,34,2026-11-06T19:59:37.959000+00:00,19.722,40.378,16.978,NaN,235.0,...,SOFT,4,18,False,False,False,76.350,Practice 2,São Paulo,13
1884,1299,11406,41,33,2026-11-06T19:59:45.713000+00:00,19.485,40.070,17.023,315.0,236.0,...,SOFT,6,20,False,False,False,75.462,Practice 2,São Paulo,14
1885,1299,11406,10,34,2026-11-06T19:59:47.181000+00:00,19.410,40.128,16.877,NaN,234.0,...,SOFT,8,18,False,False,False,75.756,Practice 2,São Paulo,10


In [16]:
lr["stint_first_lap"] = lr.groupby(["session_key", "driver_number", "stint_number"])["lap_number"].transform("min")
lr["laps_into_stint"] = lr["lap_number"] - lr["stint_first_lap"]

lr["corrected"] = lr["lap_duration"] - FUEL * lr["laps_into_stint"] - DEG * lr["tyre_age"]

In [17]:
pace = lr.groupby(["session_key", "location", "driver_number"]).agg(
    fp2_pace=("corrected", "median"),
    n_laps=("corrected", "size"),
).reset_index()

In [ ]:
pace["session_best"] = pace.groupby("session_key")["fp2_pace"].transform("min") #best pace in the session

pace["fp2_gap_pct"] = (pace["fp2_pace"] / pace["session_best"] - 1) * 100

pace.to_csv("../data/processed/fp2_pace_2026.csv", index = False)

## Race correction: fuel and tyre effects

**Model.** For each 2026 race I fitted a linear regression:
lap_time = driver_base + fuel × lap_number + deg × tyre_age.
Driver dummies (one-hot) absorb the pace difference between cars, so the
two remaining coefficients only measure fuel burn and tyre wear.

**Why races and not FP2.** In an FP2 stint, lap_number and tyre_age grow
together (+1 every lap), so the regression cannot separate them
(collinearity). In a race, tyre_age can be reset after a pit stop.

**Results (median over 21 races).**
- fuel = -0.056 s/lap: means that the driver is faster every lap with 0.056s.
- deg  = +0.030 s/lap of tyre age: means that the pilot is slower every lap with 0.030s.
- I use the median, not the mean, because a strange race should not have a huge impact. 